In [1]:
import warnings
import tensorflow as tf
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

import numpy as np
import pandas as pd
import deepchem as dc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_max_pool
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from sklearn import metrics

Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading some Jax models, missing a dependency. No module named 'jax'


In [2]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = GCNConv(30, 256)
        self.conv2 = GCNConv(256, 256)
        self.conv3 = GCNConv(256, 256)
        self.conv4 = GCNConv(256, 256)
        self.fc1 = nn.Linear(256, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)
        self.dropout1 = nn.Dropout(p=0.2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = self.conv4(x, edge_index)
        x = F.relu(x)
        x = global_max_pool(x, data.batch)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

In [3]:
def custom_collate(batch):
    data_list, target_list = zip(*batch)
    batch_data = Batch.from_data_list(data_list)
    batch_target = torch.stack(target_list)
    return batch_data, batch_target

In [4]:
def calculate_statistics(group):
    r2_test = group['r2_test']
    r2_test_dict = {f'run{i}': r2_test_val for i, r2_test_val in enumerate(r2_test)}
    return pd.Series({
        **r2_test_dict, 
        'r2_test_mean': np.mean(r2_test),
        'r2_test_max': np.max(r2_test),
        'r2_test_min': np.min(r2_test),
        'r2_test_std': np.std(r2_test, ddof=0),
    })

def calculate_statistics2(group):
    rmse_test = group['rmse_test']
    rmse_test_dict = {f'run{i}': rmse_test_val for i, rmse_test_val in enumerate(rmse_test)}
    return pd.Series({
        **rmse_test_dict, 
        'rmse_test_mean': np.mean(rmse_test),
        'rmse_test_max': np.max(rmse_test),
        'rmse_test_min': np.min(rmse_test),
        'rmse_test_std': np.std(rmse_test, ddof=0),
    })

In [5]:
torch.manual_seed(0)

epochs = 40
lr = 1e-3
wd = 1e-3

results_r2 = []
results_rmse = []
for random_state in range(10):
    torch.manual_seed(0)
    
    for dataset in ["abcgg", "aatsc3d", "atsc3d", "kappa2", "peoevsa6", "bertzct", "ggi10", "vsaestate3",
                    "atsc4i", "bcutp1l", "kappa3", "estatevsa3", "kier3", "aats8p", "kier2", "frnh0"]:
        torch.manual_seed(0)
        
        for t in ["Yield_CS"]:
            torch.manual_seed(0)
            scaler = StandardScaler()
            df = pd.read_csv('data_Real/data_real.csv')
            smiles = df["SMILES"]
            featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
            X = featurizer.featurize(smiles)
            
            y = df[t]
            data_train, data_test, target_train, target_test = train_test_split(X, y, test_size=0.5, random_state=random_state)

            target_train = scaler.fit_transform(target_train.values.reshape(-1, 1)).flatten()
            target_test = scaler.transform(target_test.values.reshape(-1, 1)).flatten()
            
            target_train = torch.tensor(target_train, dtype=torch.float32)
            target_test = torch.tensor(target_test, dtype=torch.float32)

            data_train_list = []
            for graph_data in data_train:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_train_list.append(data)

            data_test_list = []
            for graph_data in data_test:
                node_features = torch.tensor(graph_data.node_features, dtype=torch.float32)
                edge_index = torch.tensor(graph_data.edge_index, dtype=torch.long)
                edge_features = torch.tensor(graph_data.edge_features, dtype=torch.float32)
                data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_features)
                data_test_list.append(data)

            train_loader = DataLoader(list(zip(data_train_list, target_train)), batch_size=len(data_train_list), collate_fn=custom_collate)
            test_loader = DataLoader(list(zip(data_test_list, target_test)), batch_size=len(data_test_list), collate_fn=custom_collate)

            model = Net()
            model.load_state_dict(torch.load(f'data_AI2+Human/model_{dataset}_sc.pth'))
            model.fc3 = nn.Linear(128, 1)
        
            model.train()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
            criterion = nn.MSELoss()
        
            for param in model.conv1.parameters():
                param.requires_grad = False
            for param in model.conv2.parameters():
                param.requires_grad = False
            for param in model.conv3.parameters():
                param.requires_grad = False
            for param in model.conv4.parameters():
                param.requires_grad = False

            device = torch.device('cpu')
            model.to(device)

            for epoch in range(epochs):
                for data, target in train_loader:
                    data = data.to(device)
                    target = target.to(device)
                    optimizer.zero_grad()
                    out = model(data)
                    loss = criterion(out, target.view(-1, 1))
                    loss.backward()
                    optimizer.step()

            model.eval()
            pred_train = []
            for data, target in train_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_train.append(out.cpu().numpy())
            pred_train = np.concatenate(pred_train)

            pred_test = []
            for data, target in test_loader:
                data = data.to(device)
                with torch.no_grad():
                    out = model(data)
                pred_test.append(out.cpu().numpy())
            pred_test = np.concatenate(pred_test)

            pred_train = scaler.inverse_transform(pred_train)
            pred_test = scaler.inverse_transform(pred_test)
            target_train = scaler.inverse_transform(target_train.numpy().reshape(-1, 1)).flatten()
            target_test = scaler.inverse_transform(target_test.numpy().reshape(-1, 1)).flatten()

            r2_test_score = metrics.r2_score(target_test, pred_test)
            rmse_test_score = metrics.root_mean_squared_error(target_test, pred_test)
            results_r2.append({'source': dataset, 'target': t, 'r2_test': r2_test_score})
            results_rmse.append({'source': dataset, 'target': t, 'rmse_test': rmse_test_score})

results_df = pd.DataFrame(results_r2)
gen_results = results_df.groupby(['source', 'target']).apply(calculate_statistics).reset_index()
results_df2 = pd.DataFrame(results_rmse)
gen_results2 = results_df2.groupby(['source', 'target']).apply(calculate_statistics2).reset_index()

In [6]:
gen_results.T.to_csv('result/result_yield_CS_r2.csv', header=False)
gen_results.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS
run0,0.370881,0.231115,0.477189,0.40668,0.253324,0.015344,0.474472,0.213746,0.368759,0.401671,0.306726,0.430372,0.452954,0.43738,0.397021,0.270182
run1,0.179322,0.064283,0.24413,0.249528,0.135832,-0.357638,0.156558,0.067545,0.121608,0.242124,0.32878,0.250514,0.243659,0.091237,0.198723,-0.094976
run2,0.158744,0.136279,0.310611,0.046527,0.12206,0.091005,0.342458,0.078484,0.180252,0.321409,0.216519,0.222219,0.293314,0.340656,0.035333,0.169559
run3,0.183487,0.230489,0.371143,0.259685,0.285975,-0.099163,0.311948,0.153107,0.250596,0.32941,0.331837,0.299343,0.33135,0.293808,0.21469,0.077681
run4,0.361663,0.029265,0.404834,0.159088,0.151924,-0.091988,0.466445,0.197912,0.340049,0.294042,0.40745,0.390519,0.429024,0.352233,0.374192,0.0355
run5,0.002207,-0.052873,-0.00958,-0.013619,-0.169321,0.001421,-0.026618,0.026733,-0.099291,-0.071066,-0.002793,0.096928,0.01577,0.013513,-0.193514,-0.096741
run6,0.331911,0.069505,0.249565,0.151771,0.143171,0.007501,0.257264,0.274552,0.181079,0.270599,0.202595,0.169242,0.164262,0.181953,0.159763,0.143618
run7,0.379089,0.224305,0.455207,0.308397,0.215115,-0.113634,0.486717,0.096049,0.376184,0.274263,0.398752,0.39447,0.375555,0.46174,0.357183,0.287394


In [7]:
gen_results2.T.to_csv('result/result_yield_CS_rmse.csv', header=False)
gen_results2.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
source,aats8p,aatsc3d,abcgg,atsc3d,atsc4i,bcutp1l,bertzct,estatevsa3,frnh0,ggi10,kappa2,kappa3,kier2,kier3,peoevsa6,vsaestate3
target,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS,Yield_CS
run0,29.197376,32.278103,26.61643,28.354486,31.808531,36.527485,26.685509,32.64064,29.246576,28.473925,30.649958,27.782598,27.226347,27.611177,28.584352,31.44739
run1,28.233782,30.147751,27.096066,26.999142,28.972218,36.314041,28.622675,30.095148,29.209681,27.131998,25.533785,26.98139,27.104494,29.710361,27.898062,32.612556
run2,37.105366,37.597519,33.589596,39.502686,37.905743,38.57032,32.804554,38.835049,36.627968,33.325485,35.808544,35.678055,34.008373,32.84948,39.733902,36.86607
run3,31.215538,30.303768,27.394611,29.723335,29.190794,36.217667,28.654972,31.790964,29.905245,28.289022,28.237778,28.916252,28.24806,29.030237,30.613281,33.176449
run4,31.627739,39.002598,30.539518,36.30098,36.455296,41.366806,28.915636,35.453091,32.158745,33.260784,30.472315,30.9046,29.912447,31.860489,31.315815,38.877144
run5,41.402031,42.529415,41.645859,41.729088,44.819645,41.418343,41.995811,40.89003,43.456806,42.895298,41.50565,39.387882,41.119686,41.166801,45.280914,43.406384
run6,31.419353,37.079762,33.299412,35.402706,35.581738,38.295254,33.128151,32.740326,34.785725,32.829414,34.325699,35.036221,35.141064,34.767151,35.235527,35.572437
run7,27.909191,31.194502,26.142576,29.45513,31.378744,37.376942,25.375286,33.674789,27.974392,30.173254,27.463705,27.561337,27.988499,25.98534,28.397245,29.899031
